In [2]:
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import gradio as gr
import base64
from io import BytesIO
from PIL import Image
from pydub import AudioSegment
from pydub.playback import play
from datetime import datetime

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
    
MODEL = "gpt-4o-mini"

openai = OpenAI()



system_message = "Tú eres un asistente útil para responder preguntas sobre la aerolinea para viajeros pobres. "
system_message += "Respuestas cortas, certeras, no más de una sentencia. "
system_message += "Siempre sé cuidadoso. Si nom sabes las respuesta, dilo."
system_message += "Cuando el usuarrio te lo indique haz una reserva del vuelo. teniendo en cuenta que el dia de hoy es " . format(datetime.now())

ticket_prices = {"londres": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

def set_date_fly(date_city):
    print(f"Tool set_date_fly called for {date_city}")
    #fecha_dt = datetime.strptime(date_city, '%Y-%m-%d')
    return date_city

price_function = {
    "name": "get_ticket_price",
    "description": "Obtenga el precio de un billete de ida y vuelta a la ciudad de destino. Llame a esto siempre que necesite saber el precio del billete, por ejemplo, cuando un cliente pregunta 'cuánto es un billete para esta ciudad'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "La ciudad a la que el cliente quiere viajar",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


reserva_function = {
    "name": "set_date_fly",
    "description": "Haz una reserva de vuelo del avion indicando la ciudad y la fecha del vuelo cunado el usuario indique 'haz una reserva del vuelo a esta ciudad para tal día', la fecha puede ser dicha asi 'el 23 de julio' , 'la proxima semana' ",
    "parameters": {
        "type": "object",
        "properties": {
             "date_city": {
                "type": "string",
                "description": "Es la fecha que el usuario indica que quiere viajar, pasala a formato fecha",
            },
        },
        "required": ["date_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function},{"type":"function","function":reserva_function}]


OpenAI API Key exists and begins sk-proj-


In [9]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [10]:
def talker(message):
    response = openai.audio.speech.create(
      model="tts-1",
      voice="onyx",    # Also, try replacing onyx with alloy
      input=message
    )
    
    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format="mp3")
    play(audio)

In [129]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    city_global = ''
    print(tool_call)
    if function_name=='get_ticket_price':
        city = arguments.get('destination_city')
        price = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": json.dumps({"destination_city": city,"price": price}),
            "tool_call_id": tool_call.id
        }
        city_global = city
        return response, city
        
    if function_name =='set_date_fly':
        date_city = arguments.get('date_city')
        dated = set_date_fly(date_city)
        response = {
            "role": "tool",
            "content": json.dumps({"date_city": dated}),
            "tool_call_id": tool_call.id
        }
        return response, city_global
  
   


    

In [4]:
def chat(history):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    image = None
    
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city  = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        image = artist(city)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # Comment out or delete the next line if you'd rather skip Audio for now..
    talker(reply)
    
    return history, image

In [5]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history += [{"role":"user", "content":message}]
        return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=[chatbot], outputs=[chatbot, image_output]
    )
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
